In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
import os

openai_client=OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://apihub.agnes-ai.com/v1")


定义一个用于与大型语言模型进行交互的函数：

In [3]:
def llm(prompt, model="agnes-2.0-flash"):
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content

In [5]:
question = 'Who is the tutor of this course?'

context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer)

I don't know.


FAQ 文档是最适合使用 Pre-defined Questions（预定义问题）的场景之一。预定义问题的核心好处有：
1. 显著提升检索准确率（最重要）
2. 避免“用户乱问”导致系统崩盘
3. 降低 Token 成本（非常现实）短、精准的问题 → 少检索 → 少拼接上下文 → 少推理
4. 回答质量更稳定（可测试、可回归）   

| 场景         | 是否推荐     |
|--------------|------------|
| 企业知识库   | ✅ 强烈推荐 |
| 客服 / FAQ   | ✅ 必须     |
| 医疗 / 法律  | ✅ 必须     |
| 合规审计     | ✅ 必须     |
| 开放域聊天   | ❌ 不适合   |
| 创意写作     | ❌ 不适合   |

In [6]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

print(documents[0])

{'id': '0e38656cfb', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'How do I submit homework?', 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}


# 常用搜索方法速查表

| 搜索方法 | 说明 | 适用场景 | 示例 |
|----------|------|----------|------|
| 关键词搜索 | 使用核心关键词查找信息 | 日常搜索、快速查资料 | `人工智能 医疗应用` |
| 精确匹配搜索 | 使用引号匹配完整短语 | 查找原文、引用、特定名称 | `"大语言模型"` |
| 布尔搜索（AND） | 同时包含多个关键词 | 缩小搜索范围 | `AI AND 医疗` |
| 布尔搜索（OR） | 包含任意一个关键词 | 扩大搜索范围 | `ChatGPT OR Claude` |
| 布尔搜索（NOT） | 排除指定关键词 | 过滤无关内容 | `苹果 -水果` |
| 站内搜索 | 只搜索指定网站 | 查询特定网站内容 | `site:github.com RAG` |
| 文件类型搜索 | 查找特定格式文件 | 获取论文、报告、文档 | `人工智能 filetype:pdf` |
| 标题搜索 | 关键词必须出现在标题中 | 提高结果相关性 | `intitle:RAG` |
| URL搜索 | 关键词出现在网址中 | 查找专题页面 | `inurl:rag` |
| 时间范围搜索 | 限定发布时间 | 查找最新资讯 | `AI Agent after:2025-01-01` |
| 语义搜索 | 使用自然语言提问 | AI搜索、问答场景 | `如何构建企业知识库？` |
| 图片搜索 | 通过图片或图片关键词搜索 | 查找图片素材、识图 | 图片上传或关键词搜索 |
| 反向图片搜索 | 以图搜图 | 查找图片来源 | 上传图片进行搜索 |
| 视频搜索 | 搜索视频内容 | 教程、演示、课程 | `LangChain 教程` |
| 学术搜索 | 搜索论文和科研资料 | 科研、技术研究 | `Transformer paper` |
| 社区搜索 | 搜索用户讨论和经验 | 产品评价、技术问题 | `RAG Reddit` |
| 企业信息搜索 | 查询企业资料 | 商业调查、市场分析 | 企业名称 + 天眼查 |
| 法律法规搜索 | 查询法律条文和案例 | 合规、法律研究 | 法律名称 + 案例 |

---

# 搜索语法速查

| 语法 | 功能 | 示例 |
|--------|--------|--------|
| `"关键词"` | 精确匹配 | `"人工智能"` |
| `-关键词` | 排除关键词 | `苹果 -水果` |
| `A OR B` | 任意匹配 | `AI OR 人工智能` |
| `A AND B` | 同时匹配 | `AI AND 医疗` |
| `site:` | 指定网站 | `site:github.com LangChain` |
| `filetype:` | 指定文件类型 | `RAG filetype:pdf` |
| `intitle:` | 标题包含关键词 | `intitle:RAG` |
| `inurl:` | URL包含关键词 | `inurl:rag` |
| `before:` | 指定时间之前 | `AI before:2025-01-01` |
| `after:` | 指定时间之后 | `AI after:2025-01-01` |

---

# 不同场景推荐搜索方式

| 场景 | 推荐方法 |
|--------|--------|
| 查概念定义 | 关键词搜索 + 语义搜索 |
| 查最新新闻 | 时间过滤 + 新闻搜索 |
| 查论文 | 学术搜索 + PDF搜索 |
| 查技术文档 | Site搜索 + GitHub搜索 |
| 查产品评价 | 社区搜索 + 视频搜索 |
| 查企业信息 | 企业数据库搜索 |
| 查法律法规 | 法律数据库搜索 |
| 企业知识库检索 | 向量搜索 + 混合搜索（Hybrid Search） |
| RAG应用 | 语义搜索 + 向量检索 + 重排序（Rerank） |

---

# AI/RAG系统中的搜索方式

| 搜索方法 | 特点 | 推荐程度 |
|----------|------|----------|
| 关键词搜索（BM25） | 精准匹配关键词 | ✅ 推荐 |
| 向量搜索（Vector Search） | 语义理解能力强 | ✅ 强烈推荐 |
| 混合搜索（Hybrid Search） | 关键词+语义结合 | ✅ 最佳实践 |
| 重排序（Rerank） | 提升结果相关性 | ✅ 推荐 |
| 图搜索（Graph Search） | 适合知识图谱场景 | ⚠️ 特定场景 |
| 全文搜索（Full-text Search） | 文档检索能力强 | ✅ 推荐 |

In [7]:
from minsearch import Index  #创建索引

index = Index(
    text_fields=["question", "section", "answer"],  #index the question, section, and answer as text (they'll be tokenized and ranked)
    keyword_fields=["course"]   #keyword for filtering
)

index.fit(documents)

In [8]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5}, #默认权重为1，调节权重大小提升更重要的字段的贡献值
    filter_dict={"course": "llm-zoomcamp"},  #限制搜索范围
    num_results=5   
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [9]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 3.0, "section": 0.1} #可以比较上面的结果看不同权重对结果的影响
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

search(question)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [10]:
def build_context(search_results):  #构建上下文
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()
print(search_results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate 

In [11]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
""" 

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

prompt = build_prompt(question, search_results)

print(prompt)



Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [16]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

def llm(instructions, user_prompt, model='agnes-2.0-flash'):
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages=messages
    )

    return response.choices[0].message.content

def rag(query, model="agnes-2.0-flash"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model )
    return answer

answer = rag("What're the main topics of this course?")
print(answer)

Based on the provided context, the main topics of the course are:

1.  **Introduction to LLMs and RAG** (covered in Module 1)
2.  **Open-Source Data Ingestion** (covered in a Workshop, specifically mentioning `dlt`)

The context also discusses practical aspects such as using OpenAI API alternatives, managing API keys, and the decision not to use Langchain initially to learn the basics.
